## Multi-Turn RL with SageMaker AI, part 3: evaluate, deploy, and clean up

Training reward going up is necessary but not sufficient. You score the base and fine-tuned models
side by side on held-out prompts with `MultiTurnRLEvaluator`, read what the model actually learned,
diagnose what still fails, see the deployment options, and tear everything down.

| # | Notebook | What you do |
|---|---|---|
| 1 | `01_build_and_deploy_agent.ipynb` | Explore the task, tools, and reward; deploy the rollout agent to Amazon Bedrock AgentCore |
| 2 | `02_train.ipynb` | Build the prompt dataset and launch a GRPO training job with `MultiTurnRLTrainer`; read MLflow |
| 3 | `03_evaluate_and_deploy.ipynb` | Compare base vs. fine-tuned with `MultiTurnRLEvaluator`, diagnose failures, deploy, clean up |

**Before you start:** a `Completed` training job from `02_train.ipynb`, and the agent still `READY`.

### 1. Configuration

In [ ]:
import os
import sys
from pathlib import Path

import boto3
from sagemaker.train.multi_turn_rl_trainer import MultiTurnRLTrainer

STACK_NAME = "mtrl-prerequisites"  # the prerequisites.yaml stack you deployed
AGENT_NAME = "sopbench_agentcore"  # the name passed to `agentcore configure --name`
PREFIX = "sopbench-mtrl/aircraft"  # S3 prefix for the prompt parquets
AGENT_DIR = Path("sopbench_agent").resolve()

# Make `!python` / `!agentcore` resolve to this kernel's environment.
os.environ["PATH"] = f"{Path(sys.executable).parent}{os.pathsep}{os.environ['PATH']}"

sess = boto3.session.Session()
REGION = sess.region_name
assert REGION, "Set AWS_REGION (or a default region) to the region you deployed the stack in."
# The training job must run in the AgentCore runtime's region: pin it for every SDK client.
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"] = REGION
account = sess.client("sts").get_caller_identity()["Account"]

stack = sess.client("cloudformation").describe_stacks(StackName=STACK_NAME)["Stacks"][0]
outs = {o["OutputKey"]: o["OutputValue"] for o in stack.get("Outputs", [])}
BUCKET = outs["PromptsBucketName"]
ROLE_ARN = outs["JobRoleArn"]
AGENTCORE_EXEC_ROLE_ARN = outs["AgentCoreExecRoleArn"]
MLFLOW_APP_ARN = outs["MlflowAppArn"]
TRAIN_DATASET = f"s3://{BUCKET}/{PREFIX}/train/aircraft_inspection_train.parquet"
VAL_DATASET = f"s3://{BUCKET}/{PREFIX}/validation/aircraft_inspection_val.parquet"


def find_agent_runtime_arn(name=AGENT_NAME):
    """ARN of the READY AgentCore runtime with exactly this name (others may share a prefix)."""
    for runtime in MultiTurnRLTrainer.list_bedrock_agentcore_runtimes(session=sess):
        if runtime["name"] == name and runtime["status"] == "READY":
            return runtime["arn"]
    return None


print(f"account={account} region={REGION} bucket={BUCKET}")

AGENT_RUNTIME_ARN = find_agent_runtime_arn()
S3_OUTPUT_PATH = f"s3://{BUCKET}/eval-output/"
assert AGENT_RUNTIME_ARN, f"No READY runtime named {AGENT_NAME} - evaluation runs rollouts against it."

### 2. Attach to your training job

In [ ]:
from itertools import islice

from sagemaker.train.agent_rft_job import AgentRFTJob

for j in islice(AgentRFTJob.get_all(sort_by="CreationTime", sort_order="Descending"), 20):
    print(f"{j.job_status:<12} {j.job_name}")

In [ ]:
JOB_NAME = "<JobName of your Completed job>"   # TODO: paste from the list above
job = MultiTurnRLTrainer.attach(job_name=JOB_NAME)
job.refresh()
print("status:", job.job_status, "| model package:", job.output_model_package_arn)

### 3. Evaluate base vs. fine-tuned in one run

Pass the completed job as `model`: the evaluator resolves the fine-tuned model package, the agent,
and the base model from it. `evaluate_base_model=True` scores **both** in one pipeline - the honest
before/after. It runs as its own job (`JobCategory=AgentRFTEvaluation`) with
`num_prompts x eval_group_size` full rollouts per model - 23 x 8 x 2 models = 368 here - so expect
several minutes.

For real pass@k you need sampling temperature > 0 **and** `eval_group_size >= max(pass_k_values)`.
pass@k counts a rollout as a success when its reward is at least `success_threshold`. The
default is 1.0, which on this reward would count an answer with only ~20% of fields right, so set
it to 5.0: a success is then a fully correct answer, the same as TSR.
The evaluator runs *your* agent and reward on held-out data: it measures generalization, not
independence from your reward - a wrong reward is wrong on held-out data too.

In [ ]:
from sagemaker.train.evaluate import MultiTurnRLEvaluator

from sopbench_agentcore.reward import FINAL_REWARD_SCALE

comparison = MultiTurnRLEvaluator(
    model=job,
    dataset=VAL_DATASET,
    s3_output_path=S3_OUTPUT_PATH + "comparison/",
    mlflow_resource_arn=MLFLOW_APP_ARN,
    role=ROLE_ARN,
    accept_eula=True,
    evaluate_base_model=True,  # base + fine-tuned, side by side
)
hp = comparison.hyperparameters
print(hp.get_info())
hp.temperature = 0.7  # >0 so samples differ
hp.eval_group_size = 8  # must be >= max(pass_k_values)
hp.pass_k_values = [1, 2, 4, 8]
hp.success_threshold = FINAL_REWARD_SCALE  # pass@k success = every field correct (5.0)

execution = comparison.evaluate()  # non-blocking
print("evaluation started:", execution.arn)

In [ ]:
execution.wait()                     # blocks while the rollouts run

### 4. Read the results

Each model is logged as its own MLflow run (`base-model-eval`, `fine-tuned-model-eval`) under the
experiment `mtrl-eval-openai-reasoning-gpt-oss-20b`. Compare them side by side:

| Metric | What it tells you |
|---|---|
| `eval/reward/pass_at_1` | **primary success metric** - probability one sample is fully correct (with `success_threshold=5.0`) |
| `eval/reward/pass_at_k` | probability at least one of *k* samples is fully correct |
| `eval/reward/mean` | average reward, in `[-1, 5]` |
| `eval/reward/zero_frac` | fraction of rollouts scoring exactly 0 - an answer with every field wrong (no answer scores -1) |
| `eval/turns/mean` | high values suggest looping |
| `eval/tokens/response_max` | pinned at the cap means truncation |

### 5. What the model actually learned

An example from an earlier run, on the **serial-number-mismatch** failure case (installed serial differs from the
expected one, weight out of tolerance, low battery):

* **Step 0, no answer (reward -1).** The base model calls the right tools in the right order and
  even detects every problem in its reasoning - but **never emits the final answer block**.
* **Step 10, reward 5.** Same archetype; now it closes the loop with a complete report:

```json
{"aircraft_ready": "FALSE", "mechanical_inspection_result": "fail",
 "electrical_inspection_result": "retest", "component_incident_response": "failed",
 "component_mismatch_response": "failed", "cross_check_reporting_response": "failed"}
```

What changed: it **closes the loop** (emits the report at all - the single biggest lift), it
**propagates failures** instead of defaulting to `"NA"`, and its reasoning is **more directed**. A
rising reward curve is usually a **format/behavior story before a knowledge story** - the base model
often knows the answer and loses the reward on the contract. Reproduce it: in the Traces tab, sort
by `reward` and compare an early low-reward trace with a late high-reward one on the same archetype.

### 6. Diagnose failures

Read failing rollouts, not just the aggregate:

* **`pass_at_1 = 0` + high `eval/turns/mean`** - looping without solving; check tool usage.
* **`pass_at_1 = 0` + low `eval/tokens/response_mean`** - empty or malformed answers; usually the
  output contract (a missing field, or no answer tags at all).
* **Training reward far above the held-out score** - the reward parser accepts what the evaluator
  rejects: the reward hack from part 1.
* **`eval/reward/mean` near -1** - the model is not answering at all (no answer tags); check the
  prompt contract and `max_turns`.
* **Many failed rollouts** (training logs, CloudWatch) - persistent empty responses, endpoint
  errors before an answer, or malformed prompts fail the rollout; the trainer retries each up to
  `rollout_max_retries`. Check the agent is `READY` and the dataset format.

Fixes usually live in the reward or the agent prompt, not in training longer.

### 7. Deploy (reference)

The job's output is a **LoRA adapter** registered as a Model Package. Two paths - shown for
reference and not run here, because a deployed model bills until you delete it. Both take the
trainer or job directly and resolve the model package, base model, and serving container for you.

**SageMaker AI real-time endpoint** (needs the full `sagemaker` package - `pip install sagemaker` -
and a GPU instance; a quota increase may apply):

```python
import uuid

from sagemaker.serve import ModelBuilder

model_builder = ModelBuilder(model=job, instance_type="ml.g6e.48xlarge")
model_builder.accept_eula = True  # gated base model
model_builder.build()
endpoint = model_builder.deploy(
    endpoint_name=f"mtrl-endpoint-{uuid.uuid4().hex[:8]}",
    instance_type="ml.g6e.48xlarge",
    initial_instance_count=1,
)
```

**Amazon Bedrock Custom Model Import** (managed inference, no endpoint to run):

```python
from sagemaker.serve.bedrock_model_builder import BedrockModelBuilder

bedrock_builder = BedrockModelBuilder(model=job)
bedrock_builder.deploy(
    imported_model_name="mtrl-finetuned-model",
    role_arn="<an IAM role Amazon Bedrock can assume to read the model artifacts>",
)
```

See [Import a customized model into Amazon Bedrock](https://docs.aws.amazon.com/bedrock/latest/userguide/model-customization-import-model.html).
Either way, re-run the evaluator against the deployed model to confirm the numbers hold.

### 8. Best practices and next steps

* **Environment:** cheap, reproducible, representative - you will run it thousands of times.
* **Evaluation first:** decide how you judge success independently of the training reward before
  you train, so you catch reward hacking early.
* **Reward:** reward the outcome, give partial credit, align the parser with the evaluator.
* **Turn budget:** cap `max_turns` / `sampling_max_tokens` so trajectories don't truncate.
* **Monitoring:** reward mean, zero-advantage groups, response length, and whether validation tracks
  training.

**Densify further.** This agent returns one scalar reward per rollout. MTRL also accepts a **list of
per-turn rewards**, paired with `advantage_method = "group_based_per_turn"`, so credit lands on the
turn that earned it. To try it, change `compute_reward` to return one reward per sampled turn (note
that in-loop resamples of empty responses are sampled turns too). Add per-turn shaping only when you
can say what a "good turn" is - over-shaping invites new reward hacks.

**Bring your own task:** swap `tools.py`, `prompts.py`, `reward.py`, and `scripts/preprocess.py` for
your domain; the trainer, evaluator, and agent structure stay the same. See also
[Best practices for multi-turn RL in Amazon SageMaker AI](https://aws.amazon.com/blogs/machine-learning/best-practices-for-multi-turn-reinforcement-learning-in-amazon-sagemaker-ai/).

### 9. Clean up

Set `CONFIRM_CLEANUP = True` and run the cells to stop this example's training job and evaluation
(if still running), destroy the agent runtime (and its ECR repository), empty the bucket, and delete the
prerequisites stack. Also delete any endpoint you deployed in section 7.

> `agentcore destroy` also deletes the execution role you passed to `agentcore configure` - here the
> stack's `AmazonBedrockAgentCoreSDKRuntime-...` role - and CloudFormation does not notice. To deploy
> the agent again later, delete and re-create the prerequisites stack first.

> **Permissions.** Besides the policies listed in part 1, cleanup needs `ecr:DeleteRepository`,
> `s3:ListBucketVersions` / `s3:DeleteObjectVersion` on the bucket (the stack's notebook policy
> grants these), and - for the stack deletion - CloudFormation and IAM delete permissions. Run the
> last cell with the identity that created the stack if your notebook role lacks those.

In [ ]:
CONFIRM_CLEANUP = False

if CONFIRM_CLEANUP:
    # Stop only what this example started - never other jobs in the account.
    if "execution" in globals():
        execution.refresh()
        if execution.status.overall_status in ("Starting", "Executing"):
            execution.stop()
            print("stopped evaluation", execution.arn)
    job.refresh()
    if job.job_status == "InProgress":
        job.stop()
        print("stopped training job", job.job_name)

In [ ]:
if CONFIRM_CLEANUP:
    !cd {AGENT_DIR} && agentcore destroy -a {AGENT_NAME} --delete-ecr-repo --force

In [ ]:
if CONFIRM_CLEANUP:
    sess.resource("s3").Bucket(BUCKET).object_versions.delete()   # versioned bucket
    cfn = sess.client("cloudformation")
    cfn.delete_stack(StackName=STACK_NAME)
    cfn.get_waiter("stack_delete_complete").wait(StackName=STACK_NAME)
    print("deleted", STACK_NAME)

Left in place on purpose: the fine-tuned model package version (delete it in the SageMaker console
under Model registry if you don't need it), the `SagemakerEvaluation-MTRLEvaluation` pipeline (shared
by all MTRL evaluations in the account), MLflow runs, and the CloudWatch log groups under
`/aws/bedrock-agentcore/runtimes/` and `/aws/codebuild/bedrock-agentcore-*`.